# BigQuery: Agentic Migration & Data Transfer (Managed MCP)

[![Open In Colab](https://colab.research.google.com/github/maruti123/partner-demos/blob/main/partner-demos-march-2026/bq_migration_mcp_demo.ipynb)](https://colab.research.google.com/github/maruti123/partner-demos/blob/main/partner-demos-march-2026/bq_migration_mcp_demo.ipynb)

This notebook shows how an agent can translate legacy SQL and manage data transfers into BigQuery using the new **BQMS** and **DTS** Managed MCP servers — no custom scripts needed.

## Demonstrates
1. BigQuery Migration Service MCP server to perform SQL translation tasks. (Release: [March 25, 2026](https://docs.cloud.google.com/release-notes#March_25_2026))
2. BigQuery Data Transfer Service remote MCP server to enable AI agents to create, manage, and run data transfers. (Release: [March 24, 2026](https://docs.cloud.google.com/release-notes#March_24_2026))


## Use Case
A migration partner needs to move data from Hive-on-GCS to BigQuery. An agent handles the work:
1.  **Translate SQL**: Convert Hive SQL to GoogleSQL using the BQMS MCP server.
2.  **Generate DDL**: Create matching `CREATE TABLE` statements for BigQuery.
3.  **Active Migration**: Create a transfer config and immediately trigger the data load.

### Requirements
- BigQuery Migration Service and Data Transfer Service APIs enabled.
- `google-adk >= 1.28.0` installed.
- Gemini 3.1 Pro (Preview) access.

In [ ]:
# 1. Setup and Authentication
%pip install "google-adk>=1.28.0" google-genai "google-cloud-bigquery[pandas]" google-cloud-bigquery-datatransfer google-cloud-storage nest-asyncio db-dtypes --quiet --index-url https://pypi.org/simple

try:
    from google.colab import auth
    auth.authenticate_user()
    print('Authenticated via Colab')
except ModuleNotFoundError:
    print('Not running in Colab — using Application Default Credentials (ADC)')

import os
import nest_asyncio
import time
import google.auth
from google.auth.transport.requests import Request

nest_asyncio.apply()

project_id = 'YOUR_PROJECT_ID'  # @param {type:"string"}
service_account = 'YOUR_SERVICE_ACCOUNT' # @param {type:"string"}
location = 'us-central1'  # @param {type:"string"}

os.environ["GOOGLE_CLOUD_PROJECT"] = project_id
os.environ["GOOGLE_CLOUD_LOCATION"] = location
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"

# Configuration
DATASET_NAME = "legacy_migration_target"
TABLE_NAME = "user_events"
BUCKET_SUFFIX = "hive-source"
DATA_FILE = "user_events/data.json"

DATASET_ID = f"{project_id}.{DATASET_NAME}"
TABLE_ID = f"{DATASET_ID}.{TABLE_NAME}"
BUCKET_NAME = f"{project_id}-{BUCKET_SUFFIX}"
SOURCE_URI = f"gs://{BUCKET_NAME}/{DATA_FILE}"

### 2. Project Configuration & Service Enablement

Both BQMS and DTS Managed MCP servers must be enabled, along with the source/target APIs.

In [ ]:
# 1. Set the default project globally
!gcloud config set project {project_id} --quiet

# 2. Enable APIs
!gcloud services enable bigquery.googleapis.com bigquerymigration.googleapis.com bigquerydatatransfer.googleapis.com storage.googleapis.com --quiet

# 3. Enable Managed MCP services
!gcloud beta services mcp enable bigquerymigration.googleapis.com --quiet
!gcloud beta services mcp enable bigquerydatatransfer.googleapis.com --quiet

print("Success: APIs and Managed MCP services enabled.")

### 3. IAM Roles Setup

To use Managed MCP servers, your identity must have the **MCP Tool User** role. We grant the required roles below.

In [ ]:
roles = [
    "roles/mcp.toolUser",
    "roles/bigquerymigration.editor",
    "roles/bigquery.admin"
]

for role in roles:
    print(f"Granting {role} to {service_account}...")
    !gcloud projects add-iam-policy-binding {project_id} --member=user:{service_account} --role={role} --condition=None --quiet > /dev/null

print("\nSuccess: IAM Roles configured. Waiting for propagation...")
time.sleep(30)

### 4. Infrastructure Prerequisites

Create an **empty** BigQuery table and upload Hive data (JSONL format) to GCS, simulating a Hive external table migration scenario.

In [ ]:
from google.cloud import bigquery, storage
from google.api_core import exceptions
import json

def create_empty_bigquery_table(bq_client):
    """Create empty BigQuery table with array schema."""
    dataset = bigquery.Dataset(DATASET_ID)
    dataset.location = location
    bq_client.create_dataset(dataset, exists_ok=True)

    schema = [
        bigquery.SchemaField("user_id", "INTEGER"),
        bigquery.SchemaField("session_id", "STRING"),
        bigquery.SchemaField("event_types", "STRING", mode="REPEATED"),
        bigquery.SchemaField("product_ids", "INTEGER", mode="REPEATED"),
        bigquery.SchemaField("dt", "DATE"),
    ]
    
    table = bigquery.Table(TABLE_ID, schema=schema)
    bq_client.create_table(table, exists_ok=True)
    
    # Ensure table is empty for demo
    bq_client.query(f"DELETE FROM `{TABLE_ID}` WHERE TRUE").result()

def upload_hive_data_to_gcs(storage_client):
    """Upload JSONL data to GCS simulating Hive external table."""
    try:
        bucket = storage_client.create_bucket(BUCKET_NAME, location=location)
    except exceptions.Conflict:
        bucket = storage_client.get_bucket(BUCKET_NAME)
    
    # Sample Hive data in JSONL format
    hive_data = [
        {
            "user_id": 1001,
            "session_id": "sess_a1",
            "event_types": ["page_view", "add_to_cart", "purchase"],
            "product_ids": [101, 102, 103],
            "dt": "2025-01-15"
        },
        {
            "user_id": 1002,
            "session_id": "sess_b2",
            "event_types": ["page_view", "search"],
            "product_ids": [201],
            "dt": "2025-01-16"
        },
        {
            "user_id": 1003,
            "session_id": "sess_c3",
            "event_types": ["page_view", "add_to_cart"],
            "product_ids": [301, 302],
            "dt": "2025-01-17"
        },
        {
            "user_id": 1004,
            "session_id": "sess_d4",
            "event_types": ["page_view", "purchase", "review"],
            "product_ids": [401, 402, 403, 404],
            "dt": "2025-01-18"
        }
    ]
    
    # Convert to JSONL (newline-delimited JSON)
    jsonl_content = '\n'.join([json.dumps(row) for row in hive_data])
    
    blob = bucket.blob(DATA_FILE)
    blob.upload_from_string(jsonl_content, content_type='application/json')
    
    return len(hive_data)

def setup_migration_prereqs():
    """Setup empty BQ table and Hive data in GCS."""
    bq_client = bigquery.Client(project=project_id, location=location)
    storage_client = storage.Client(project=project_id)
    
    create_empty_bigquery_table(bq_client)
    row_count = upload_hive_data_to_gcs(storage_client)
    
    print(f"✅ Migration prerequisites ready:")
    print(f"   Empty table: '{TABLE_ID}'")
    print(f"   Hive data in GCS: '{SOURCE_URI}' ({row_count} rows)")
    print(f"   Format: JSONL (newline-delimited JSON)")
    
    return bq_client

bq_client = setup_migration_prereqs()

### 5. Initialize Managed MCP Toolsets

We initialize the toolsets for BigQuery Migration (BQMS) and Data Transfer (DTS).

In [ ]:
from google.adk.tools.mcp_tool import McpToolset, StreamableHTTPConnectionParams

scopes = ["https://www.googleapis.com/auth/cloud-platform"]
creds, _ = google.auth.default(scopes=scopes)

# Only refresh if token is invalid or expired
if not creds.valid:
    if creds.expired and creds.refresh_token:
        creds.refresh(Request())

bqms_mcp = McpToolset(
    connection_params=StreamableHTTPConnectionParams(
        url="https://bigquerymigration.googleapis.com/mcp",
        headers={"Authorization": f"Bearer {creds.token}"},
        timeout=60.0
    )
)

dts_mcp = McpToolset(
    connection_params=StreamableHTTPConnectionParams(
        url="https://bigquerydatatransfer.googleapis.com/mcp",
        headers={"Authorization": f"Bearer {creds.token}"},
        timeout=60.0
    )
)
print("Managed MCP Toolsets initialized.")

### 6. Define the Migration Assistant Agent

We create a specialized agent equipped with the migration toolsets. We give it explicit instructions to trigger the transfer run.

In [ ]:
from google.adk import Agent, Runner
from google.adk.sessions.in_memory_session_service import InMemorySessionService

os.environ["GOOGLE_CLOUD_LOCATION"] = "global"

migration_agent = Agent(
    model="gemini-3.1-pro-preview",
    name="MigrationAssistant",
    instruction="""
    You are a data migration expert using BQMS for SQL translation and DTS for data transfers.
    When asked to move data, create the transfer config and immediately trigger execution.
    """,
    tools=[bqms_mcp, dts_mcp]
)

runner = Runner(
    agent=migration_agent,
    session_service=InMemorySessionService(),
    app_name="migration_mcp_demo",
    auto_create_session=True
)
print("Migration Assistant Agent and Runner initialized.")

### 7. Run Agentic Migration Workflow

The agent will:
1. **Translate** HiveQL to GoogleSQL using BQMS MCP
2. **Transfer** data from GCS to BigQuery using DTS MCP

In [ ]:
from google.genai import types
import traceback

async def run_migration_workflow():
    legacy_hiveql = """
    SELECT
      user_id,
      COLLECT_SET(product_id) as unique_products,
      COUNT(DISTINCT session_id) as session_count
    FROM (
      SELECT
        user_id,
        event_type,
        product_id,
        session_id,
        dt
      FROM user_events
      LATERAL VIEW EXPLODE(event_types) et AS event_type
      LATERAL VIEW EXPLODE(product_ids) pt AS product_id
      WHERE dt > '2025-01-01'
    ) exploded_data
    GROUP BY user_id
    ORDER BY user_id
    LIMIT 100;
    """
    
    prompt = f"""
    You are migrating Hive data to BigQuery. Complete these tasks in order:

    TASK 1 - TRANSLATE SQL:
    Translate this HiveQL query to GoogleSQL:
    {legacy_hiveql}

    TASK 2 - TRANSFER DATA:
    After translation, use DTS to transfer data from GCS to BigQuery:
    - Source: {SOURCE_URI} (JSONL format)
    - Destination table: {TABLE_ID}
    - Service account: {service_account}
    - CRITICAL: Create the transfer config and IMMEDIATELY trigger the run.

    Complete both tasks.
    """

    print("=" * 80)
    print("AGENTIC MIGRATION WORKFLOW")
    print("=" * 80)
    print(f"\nStep 1: Translate HiveQL to GoogleSQL")
    print(f"Step 2: Transfer data from GCS to BigQuery using DTS")
    print("\nOriginal HiveQL:")
    print(legacy_hiveql)
    print("=" * 80)

    try:
        message = types.Content(parts=[types.Part(text=prompt)], role='user')
        async for event in runner.run_async(
            user_id="partner_user",
            session_id="migration_session",
            new_message=message
        ):
            if event.content and event.content.parts:
                for part in event.content.parts:
                    if part.text:
                        print(f"\n🤖 Agent: {part.text}")
                    if part.function_call:
                        print(f"[TOOL]: {event.author} calling '{part.function_call.name}'")
    except Exception as e:
        print(f"\n❌ [ERROR]: Workflow failed.")
        if isinstance(e, BaseExceptionGroup):
            for sub_e in e.exceptions:
                print(f"  Sub-exception: {sub_e}")
        traceback.print_exc()

await run_migration_workflow()

### 8. [VERIFICATION] Test Translated Query

Verify the data is loaded and test the translated GoogleSQL query on real BigQuery data.

In [ ]:
import time

print("\n" + "=" * 80)
print("VERIFICATION")
print("=" * 80)

print("\nWaiting 30s for DTS transfer to complete...")
time.sleep(30)

print("\n1️⃣ Verifying data transferred from GCS to BigQuery...")
count_query = f"SELECT COUNT(*) as row_count FROM `{TABLE_ID}`"

try:
    result = bq_client.query(count_query).result()
    row_count = next(result).row_count
    
    if row_count == 0:
        print("❌ No data found. DTS transfer may still be processing.")
        print("   Check DTS console or wait longer and re-run this cell.")
    else:
        print(f"✅ Success! Found {row_count} rows transferred from GCS")
        
        # Show sample data
        sample_query = f"SELECT * FROM `{TABLE_ID}` LIMIT 5"
        df = bq_client.query(sample_query).to_dataframe()
        print("\nSample data with arrays:")
        display(df)
        
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()

print("\n2️⃣ Testing translated GoogleSQL query...")
print("   HiveQL: LATERAL VIEW EXPLODE → GoogleSQL: UNNEST")
print("   HiveQL: COLLECT_SET → GoogleSQL: ARRAY_AGG(DISTINCT)\n")

translated_googlesql = f"""
SELECT
  user_id,
  ARRAY_AGG(DISTINCT product_id) as unique_products,
  COUNT(DISTINCT session_id) as session_count
FROM (
  SELECT
    user_id,
    event_type,
    product_id,
    session_id,
    dt
  FROM `{TABLE_ID}`,
  UNNEST(event_types) AS event_type,
  UNNEST(product_ids) AS product_id
  WHERE dt > '2025-01-01'
) exploded_data
GROUP BY user_id
ORDER BY user_id
LIMIT 5
"""

try:
    df_result = bq_client.query(translated_googlesql).to_dataframe()
    print("✅ Translated query executed successfully!\n")
    print("Results (arrays unnested and aggregated):")
    display(df_result)
except Exception as e:
    print(f"❌ Query failed: {e}")
    print("This query requires data in the table from DTS transfer.")
    import traceback
    traceback.print_exc()

### 9. Summary of Results

**What This Demo Shows:**
- ✅ **SQL Translation (BQMS MCP)**: Agent translates HiveQL → GoogleSQL
  - `LATERAL VIEW EXPLODE` → `UNNEST`
  - `COLLECT_SET` → `ARRAY_AGG(DISTINCT)`
- ✅ **Data Transfer (DTS MCP)**: Agent transfers GCS → BigQuery
  - Source: JSONL in GCS (simulates Hive external table)
  - Destination: BigQuery table with REPEATED columns
- ✅ **End-to-End Workflow**: Translation first, then data transfer
- ✅ **No Custom Code**: Agent uses Managed MCP servers

**Demo Flow:**
1. Create empty BigQuery table
2. Upload Hive data to GCS (JSONL format)
3. Agent translates HiveQL using BQMS MCP
4. Agent transfers data using DTS MCP
5. Verify data arrival and test translated query

**Availability**: Preview as of March 24-25, 2026.